## load it

https://huggingface.co/alireza-2003/bert-fa-base-uncased-for-discrepancy-detection

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
import torch
import os
import pandas as pd
import re
import emoji

from dotenv import load_dotenv
load_dotenv()  

In [ ]:
data_path = os.getenv('DATA_PATH')
comments = pd.read_csv(f'{data_path}/all_labeled_comments.csv')
comments_train_data = pd.read_csv(f"{data_path}/labeled_comments.csv")


def preprocessing(comment):
    """
    Preprocesses a given comment by performing the following steps:
    1. Replaces all emojis with a space.
    2. Removes URLs.
    3. Removes all non-word characters (punctuation).
    4. Removes all digits.

    Parameters:
    comment (str): The input comment to be preprocessed.

    Returns:
    str: The preprocessed comment.
    """
    comment = emoji.replace_emoji(comment, replace=" ")
    comment = re.sub(r'https?://\S+|www\.\S+', ' ', comment)
    comment = re.sub(r'[^\w\s]', ' ', comment)
    comment = re.sub(r'[^\w\s\n]', ' ', comment)
    return comment

df = comments[comments['delivery']=='False'].sample(1000)
df['preprocessed_comment'] = df['description'].apply(preprocessing)

In [ ]:
# load model
tokenizer = AutoTokenizer.from_pretrained('alireza-2003/bert-fa-base-uncased-for-discrepancy-detection')
model = AutoModelForSequenceClassification.from_pretrained('alireza-2003/bert-fa-base-uncased-for-discrepancy-detection')

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval() 

In [ ]:
batch_size = 32

def predict_labels(texts, batch_size=64):
    all_preds = []

    with torch.no_grad():  
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]  
            inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
            inputs = {key: val.to(device) for key, val in inputs.items()} 

            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, axis=1).cpu().numpy()
            all_preds.extend(preds)

    return all_preds


df["predicted_label"] = predict_labels(df["preprocessed_comment"].tolist(), batch_size=batch_size)

label_map = {0: "Not a complaint", 1: "Discrepancy complaint"}
df["predicted_label"] = df["predicted_label"].map(label_map)